In [1]:
import os
from nwtrace import *
import pandas as pd
import geopandas as gpd

from pathlib import Path


In [2]:
lines = Path('data/more/full_sewers.geojson')
nodes = Path('data/more/all_node_connections.geojson')

lines_gdf = gpd.read_file(lines)
nodes_gdf = gpd.read_file(nodes)

nodes_gdf = nodes_gdf.to_crs(lines_gdf.crs)


In [3]:
multiple = True
upstream_only = False
downstream_only = False
verbose = True

sewer_id_field = 'FACILITYID'
upstream_field = 'FROMMH'
downstream_field = 'TOMH'

outfall_file = 'data/more/BC_outfalls.csv'
id_field = 'Asset Identification'

outfalls = pd.read_csv(outfall_file)[id_field].tolist()

target_endpoints = outfalls

outputname_extra = "allBC_"
output_dir = f"./out"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

result = []

sewershed = NWTrace(
    network=lines_gdf,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
)

# additional connections
fittings = gpd.read_file("data/more/fitting_connections.geojson")
nodes_up = (fittings[["FACILITYID", "TO_FIXED"]]
            .dropna(subset=["FACILITYID", "TO_FIXED"]) # remove rows with None values
            .rename(columns={"FACILITYID": 'node_id', "TO_FIXED": 'segment_id'})
            .to_dict(orient="records"))

sewershed.add_upstream_nodes(nodes_up)


node_lookup, seg_lookup = sewershed.get_lookup_tables()
dir_node_lookup, dir_seg_lookup = sewershed.get_directional_lookup_tables()


Added 5245 node-segment connection(s)
Created 627 new node(s)
Created 374 new segment(s).



In [4]:
errors = utils.verify_network_geometry(
    lines=lines_gdf,
    points=nodes_gdf,
    segment_lookup=seg_lookup,
    line_id_field="FACILITYID",
    point_id_field="FACILITYID",
    threshold=100
)

err_df = gpd.GeoDataFrame.from_dict(errors, geometry="geometry")
err_df = err_df.set_crs(lines_gdf.crs)

err_df.to_file("out/errors.gpkg", driver="GPKG", layer="errorsv1")

err_df.value_counts("error_t")

100%|██████████| 169323/169323 [00:04<00:00, 34897.19it/s]


1176 Errors Found
Average Distance 0.14510461119443013 units


error_t
missing node       789
missing segment    374
spatial             13
Name: count, dtype: int64

In [5]:
ids_dup = utils.count_duplicates(lines_gdf, "FACILITYID", minimum_count=1)
ids_dup

{'duplicate_count': {}}

In [12]:
lines_gdf_indexed = utils.repair_spatial_errors(
    errors=errors,
    lines=lines_gdf,
    points=nodes_gdf,
    line_id_field="FACILITYID",
    upstream_field="FROMMH",
    downstream_field="TOMH",
    distance_threshold=1
)

lines_gdf_indexed

,OBJECTID,FACILITYID,WATERTYPE,STATUS,OWNEDBY,FROMMH,TOMH,LOCDESC,WARD,INSTALLDATE,...,DROP_YN,DROP_SIZE,DROP_INVERT,SOURCE_ENG_DWG,TRUNK_NAME,TRUNK_SEWER,DISTRICT,TWIN_NUMBER,SHAPE_Length,geometry
FACILITYID,,,,,,,,,,,,,,,,,,,,,
SL9307,1,SL9307,Storm,ACTV,City,MH3848606859,MH3847906855,KEELE ST,None,1983-01-01 00:00:00+00:00,...,None,NaN,NaN,SEW.353-18,None,No,D2 - Etobicoke - York,1,8.799450,"MULTILINESTRING ((306875.532 4838709.372, 3068..."
SL9320,2,SL9320,Storm,ACTV,City,MH3847906855,MH3839806880,KEELE ST,None,1983-01-01 00:00:00+00:00,...,None,NaN,NaN,SEW.353-18,None,No,D2 - Etobicoke - York,1,85.581822,"MULTILINESTRING ((306871.024 4838701.815, 3068..."
SL1402455,3,SL1402455,Storm,ACTV,City,MH4252412459,CN8395,LAWRENCE AVE W,None,1972-01-01 00:00:00+00:00,...,None,NaN,NaN,L-207,None,No,D3 - North York,1,1.315146,"MULTILINESTRING ((312475.268 4842746.01, 31247..."
SL1402456,4,SL1402456,Storm,ACTV,City,MH4253112486,CN8396,LAWRENCE AVE W,None,1972-01-01 00:00:00+00:00,...,None,NaN,NaN,L-207,None,No,D3 - North York,1,1.544929,"MULTILINESTRING ((312502.004 4842753.76, 31250..."
SL24393,5,SL24393,Combined,ACTV,City,MH3803706077,MH3802706070,LAMBTON AVE,None,1985-01-01 00:00:00+00:00,...,None,NaN,NaN,SEW.72-1,None,No,D2 - Etobicoke - York,1,12.226952,"MULTILINESTRING ((306093.485 4838259.069, 3060..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SL1482627,168945,SL1482627,SAN,ACTV,City,JP103425,JP100680,BATHURST ST,None,1977-01-01 00:00:00+00:00,...,None,NaN,NaN,SA-256_09,None,No,D1 - Toronto-East York,1,498.810430,"MULTILINESTRING ((313096.62 4832697.137, 31302..."
SL1458848,168946,SL1458848,SAN,ACTV,City,MH3382208001,MH5512534191,ALLOTMENT LANE*EAS,None,1980-01-01 00:00:00+00:00,...,None,NaN,NaN,"1021-D-10451, 1021-D-10453",None,No,D2 - Etobicoke - York,1,837.094224,"MULTILINESTRING ((308018.622 4834046.097, 3080..."
SL1458849,168947,SL1458849,SAN,ACTV,City,MH3382208001,MH5512534191,ALLOTMENT LANE*EAS,None,1980-01-01 00:00:00+00:00,...,None,NaN,NaN,"1021-D-10451, 1021-D-10453",None,No,D2 - Etobicoke - York,2,838.263031,"MULTILINESTRING ((308018.622 4834046.097, 3080..."


In [23]:
line_id_field = "FACILITYID"
points = nodes_gdf
lines = lines_gdf
distance_threshold = 1

# Extract spatial errors from the error list
spatial_err_segs = [err["segment_id"] for err in errors if err.get("error_t") == "spatial"]
spatial_err_nodes = [err["node_id"] for err in errors if err.get("error_t") == "spatial"]

# Select segments from the lines dataset that appear in the list of spatial errors
segs = lines.loc[lines[line_id_field].isin(spatial_err_segs)]

# Get the endpoints of each segment with a spatial error, add to a dataset of endpoints
nearby_nodes = utils.find_nearby_nodes(segs, points, line_id_field, distance_threshold=distance_threshold)

# Allow easy searching of lines dataset for connecting nodes
current_from = lines.set_index(line_id_field)[upstream_field]
current_to = lines.set_index(line_id_field)[downstream_field]

# returns true when a row from 'nearby_nodes' is not connected to the associated segment in the lookup table
def not_already_connected(row):
    seg = row["segment_id"]
    role = row["role"]
    node = row[line_id_field]  # or whatever field
    
    if role == "from":
        return node != current_from.get(seg)
    else:
        return node != current_to.get(seg)
    
# apply the above funtion to nearby nodes (extaract all nodes that arent connected to segments in the lookup table)
nearby_nodes = nearby_nodes[
    nearby_nodes.apply(not_already_connected, axis=1)
]

# get only proposed node-segment connectioned with the minimum distance
idx = (
    nearby_nodes
    .groupby(["segment_id", "role"])["dist"]
    .idxmin()
)

# get a subset of nearby nodes containing only the potential connections witht the closest endpoint-node distance
best_candidates = nearby_nodes.loc[idx].drop_duplicates()[["FACILITYID", "segment_id", "role"]]

best_candidates = best_candidates.pivot(
    index="segment_id",
    columns="role",
    values=line_id_field
)

best_candidates = best_candidates.rename(
    columns={
        "from": upstream_field,
        "to": downstream_field
    }
)

lines_gdf_indexed = lines.set_index(line_id_field, drop=False)
lines_gdf_indexed.update(best_candidates)

In [25]:
segs

,OBJECTID,FACILITYID,WATERTYPE,STATUS,OWNEDBY,FROMMH,TOMH,LOCDESC,WARD,INSTALLDATE,...,DROP_YN,DROP_SIZE,DROP_INVERT,SOURCE_ENG_DWG,TRUNK_NAME,TRUNK_SEWER,DISTRICT,TWIN_NUMBER,SHAPE_Length,geometry
1324,1325,SL202006,Storm,ACTV,City,MH3004490,MH3004511,PATRICIA AVE,Willowdale (18),2012-01-01 00:00:00+00:00,...,No,NaN,NaN,U-6386-007 TO U-6386-012,None,No,D3 - North York,1,31.484215,"MULTILINESTRING ((310223.864 4849674.085, 3102..."
4941,4942,SL2008024,SAN,ACTV,City,MH2893004953,MH2893004953A,LAKE SHORE BLVD W,None,NaT,...,None,NaN,NaN,None,(324) Lakeshore STS,Yes,D2 - Etobicoke - York,1,46.086969,"MULTILINESTRING ((304969.545 4829151.995, 3050..."
6088,6089,SL4030218,Storm,ACTV,City,MH5001912133,MH5512527518,NORTHWOOD DR,None,1976-01-01 00:00:00+00:00,...,None,NaN,NaN,N-29-02,None,No,D3 - North York,1,26.840239,"MULTILINESTRING ((312149.609 4850242.027, 3121..."
6127,6128,SL4001667,Storm,ACTV,City,MH4540901031,MH4540901031A,WW N LANYARD E LINDYLOU,None,1960-01-01 00:00:00+00:00,...,None,NaN,NaN,L-147,None,No,D2 - Etobicoke - York,1,143.305489,"MULTILINESTRING ((301051.322 4845625.605, 3010..."
88153,88154,SL2000980,Storm,ACTV,City,MH3241498248,MH3237498275,BURNT LOG CRES,None,1964-01-01 00:00:00+00:00,...,None,NaN,NaN,PSB-2567-2,None,No,D2 - Etobicoke - York,1,48.429354,"MULTILINESTRING ((298264.734 4832636.572, 2982..."
159500,159501,SL110439,Storm,ACTV,City,MH2010455,MH2010446,JANE ST,York South-Weston (5),2015-01-01 00:00:00+00:00,...,No,NaN,NaN,"M-694-007, U-694-004 to 006 & U-694-008",None,No,D2 - Etobicoke - York,1,87.174978,"MULTILINESTRING ((304220.987 4841714.053, 3042..."
159573,159574,SL110441,SAN,ACTV,City,MH2010420,MH2010413,MAPLE BLVD,Etobicoke-Lakeshore (3),2007-01-01 00:00:00+00:00,...,No,NaN,NaN,U-2049-001,None,No,D2 - Etobicoke - York,1,116.092231,"MULTILINESTRING ((304915.635 4829264.625, 3048..."
159615,159616,SL110440,Storm,ACTV,City,MH2010446,MH2010447,JANE ST,York South-Weston (5),2015-01-01 00:00:00+00:00,...,No,NaN,NaN,"M-694-007, U-694-004 to 006 & U-694-008",None,No,D2 - Etobicoke - York,1,17.911008,"MULTILINESTRING ((304205.224 4841799.791, 3041..."
160234,160235,SL1426097-1,CSO,ACTV,City,MH3584318439,JP3572118478,LESLIE ST,Toronto-Danforth (14),1925-01-01 00:00:00+00:00,...,None,NaN,NaN,J-33_1,None,No,D1 - Toronto-East York,1,15.174447,"MULTILINESTRING ((318489.389 4835958.492, 3184..."
160437,160438,SL110922,Storm,ACTV,City,MH2010967,MH2010970,DUNDAS-KIPLING RAMP,Etobicoke-Lakeshore (3),2021-01-01 00:00:00+00:00,...,No,NaN,NaN,U-558-072,None,No,D2 - Etobicoke - York,1,51.341813,"MULTILINESTRING ((302025.529 4833627.248, 3020..."


In [7]:
# Regenerate lookup tables with the repaired dataset
fixed_sewershed = NWTrace(
    network=lines_gdf_indexed,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
)

# additional connections
fixed_sewershed.add_upstream_nodes(nodes_up)


f_node_lookup, f_seg_lookup = fixed_sewershed.get_lookup_tables()
f_dir_node_lookup, f_dir_seg_lookup = fixed_sewershed.get_directional_lookup_tables()

Added 5245 node-segment connection(s)
Created 627 new node(s)
Created 374 new segment(s).



In [8]:
# Check the amount of spatial errors in the newly repaired dataset
f_errors = utils.verify_network_geometry(
    lines=lines_gdf_indexed,
    points=nodes_gdf,
    segment_lookup=f_seg_lookup,
    line_id_field="FACILITYID",
    point_id_field="FACILITYID",
    threshold=100
)

f_err_df = gpd.GeoDataFrame.from_dict(f_errors, geometry="geometry")
f_err_df = f_err_df.set_crs(lines_gdf.crs)

f_err_df.value_counts("error_t")

100%|██████████| 169323/169323 [00:02<00:00, 77704.90it/s]

1171 Errors Found
Average Distance 0.07735004100557342 units


error_t
missing node       789
missing segment    374
spatial              8
Name: count, dtype: int64

In [9]:
f_err_df[f_err_df["error_t"] == "spatial"]

,node_id,segment_id,error_t,error_msg,dist,geometry
33,CN4931,SL110441,spatial,node CN4931 is more than 100 units from segmen...,12449.602404,"MULTILINESTRING ((304915.635 4829264.625, 3048..."
146,CN15964,SL2000980,spatial,node CN15964 is more than 100 units from segme...,10005.862926,"MULTILINESTRING ((298264.734 4832636.572, 2982..."
211,CN4289,SL2008024,spatial,node CN4289 is more than 100 units from segmen...,109.032942,"MULTILINESTRING ((304969.545 4829151.995, 3050..."
235,CN5024,SL202006,spatial,node CN5024 is more than 100 units from segmen...,574.136845,"MULTILINESTRING ((310223.864 4849674.085, 3102..."
236,CN16295,SL202006,spatial,node CN16295 is more than 100 units from segme...,579.284318,"MULTILINESTRING ((310223.864 4849674.085, 3102..."
546,CN4796,SL4001667,spatial,node CN4796 is more than 100 units from segmen...,305.233543,"MULTILINESTRING ((301051.322 4845625.605, 3010..."
578,CN4615,SL4030218,spatial,node CN4615 is more than 100 units from segmen...,130.062911,"MULTILINESTRING ((312149.609 4850242.027, 3121..."
599,CN5321,SL4050884,spatial,node CN5321 is more than 100 units from segmen...,215.351764,"MULTILINESTRING ((317398.129 4847530.434, 3173..."
